# CAR PRICE PREDICTION PROJECT 

In [ ]:
# importing libraries

import numpy as np 
import pandas as pd 
import seaborn as sns 
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# loading dataset

df = pd.read_csv("D:\machine learning\car_price_dataset.csv")
df

# EDA

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
# target variable exploration

In [ ]:
sns.histplot(df['price'], kde = True)

In [ ]:
# analyzing numerical features

In [ ]:
num_col = ['wheelbase', 'carlength', 'carwidth', 'carheight', 'enginesize', 'horsepower', 'citympg', 'highwaympg']

for col in num_col:
    plt.figure(figsize=(8,5))
    sns.histplot(df[col], kde=True)

In [ ]:
# analyzing categorical features

In [ ]:
cat_col = ['fueltype', 'fuelsystem', 'aspiration', 'enginetype', 'doornumber', 'carbody', 'drivewheel', 'cylindernumber']

for col in cat_col:
    plt.figure(figsize=(8,5))
    sns.barplot(x=df[col], y=df['price'], palette='coolwarm')

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(df.corr(numeric_only=True), annot=True)

In [ ]:
# creating X(input features) and y(output features) for model

In [ ]:
X = df.drop(columns = ['price'], axis=1)
y = pd.DataFrame(df['price'])                     # converting y to dataframe otherwise it will give series

# Data Cleaning & Presprocessing

In [ ]:
# feature engineering

In [ ]:
X['symboling_category'] = pd.cut(X['symboling'], bins = [-3,-1,1, float('inf')], labels=['very_safe', 'avg_risk', 'high_risk'])
X.head()

In [ ]:
X['horsepower_category'] = pd.cut(X['horsepower'], bins = [0, 100, 150, 250, float('inf')], labels = ['economy', 'standard', 'sports', 'supercar'])
X.head()

In [ ]:
X['enginesize_category'] = pd.cut(X['enginesize'], bins = [0, 120, 200, float('inf')], labels = ['small', 'medium', 'large'])
X.head()

In [ ]:
# encoding X

In [ ]:
# one hot encoding

X_encode = pd.get_dummies(X, drop_first=True).astype(int)
X_encode.head()

In [ ]:
# standard scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

num_col = ['symboling', 'wheelbase', 'carlength', 'carwidth', 'carheight', 'enginesize', 'horsepower', 'citympg', 'highwaympg']

X_encode[num_col] = scaler.fit_transform(X_encode[num_col])
X_encode.head()

In [ ]:
# pearson correlation for numerical features

In [ ]:
from scipy.stats import pearsonr

columns = ['symboling', 'wheelbase', 'carlength', 'carwidth', 'carheight', 'enginesize', 'horsepower', 'citympg', 'highwaympg']

correlations = {
    feature: pearsonr(X_encode[feature], y['price'])[0]
    for feature in columns
}

In [ ]:
corr = pd.DataFrame(list(correlations.items()), columns = ['Feature', 'Pearson Correlation']).sort_values(by='Pearson Correlation', ascending=False)
corr

In [ ]:
# chi2 test for categorical features

In [ ]:
from scipy.stats import chi2_contingency

cat_col = ['fueltype_gas', 'aspiration_turbo', 'doornumber_two', 'carbody_hardtop', 'carbody_hatchback', 'carbody_sedan', 'carbody_wagon', 'drivewheel_fwd',
           'drivewheel_rwd', 'enginetype_dohcv', 'enginetype_l', 'enginetype_ohc', 'enginetype_ohcf', 'enginetype_ohcv', 'enginetype_rotor',
           'cylindernumber_five', 'cylindernumber_four', 'cylindernumber_six', 'cylindernumber_three', 'cylindernumber_twelve', 'cylindernumber_two',
           'fuelsystem_2bbl', 'fuelsystem_4bbl', 'fuelsystem_idi', 'fuelsystem_mfi', 'fuelsystem_mpfi', 'fuelsystem_spdi', 'fuelsystem_spfi', 
           'symboling_category_avg_risk', 'symboling_category_high_risk', 'horsepower_category_standard', 'horsepower_category_sports', 'horsepower_category_supercar',
           'enginesize_category_medium', 'enginesize_category_large']
alpha = 0.05

In [ ]:
# creating bins in price

y['price_bins'] = pd.qcut(y['price'], q=4, labels=False)
y

In [ ]:
chi2_results = {}

for col in cat_col:
    contingency = pd.crosstab(X_encode[col], y['price_bins'])
    chi2_stats, p_val, _, _ = chi2_contingency(contingency)
    decision = "Reject Null(Keep Feature)" if p_val < alpha else "Accept Null(Drop Feature)"
    chi2_results[col] = {
        'chi2_statistic': chi2_stats,
        'p_value': p_val,
        'Decision': decision
    }

In [ ]:
chi2_df = pd.DataFrame(chi2_results).T.sort_values(by='p_value', ascending=True)
chi2_df

In [ ]:
# removing less correlated features

X_final = X_encode[['enginesize', 'horsepower', 'carwidth', 'carlength', 'wheelbase', 'fuelsystem_2bbl', 'drivewheel_rwd', 'drivewheel_fwd', 
                    'fuelsystem_mpfi', 'cylindernumber_four', 'enginesize_category_medium', 'horsepower_category_sports', 'horsepower_category_standard', 
                    'cylindernumber_six', 'enginesize_category_large', 'enginetype_ohc', 'aspiration_turbo', 'cylindernumber_five', 
                    'symboling_category_avg_risk', 'enginetype_ohcv', 'carbody_hatchback', 'cylindernumber_two', 'enginetype_rotor', 
                    'symboling_category_high_risk', 'fuelsystem_4bbl', 'enginetype_l']]
X_final.head()

# Linear Regression Model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

In [ ]:
y = y['price']

- First Model

In [ ]:
# data split

X_train, X_test, y_train, y_test = train_test_split(X_final, y, test_size=0.20, random_state=42)

In [ ]:
# training

model1 = LinearRegression()
model1.fit(X_train, y_train)

In [ ]:
# testing

y_pred = model1.predict(X_test)
y_pred

In [ ]:
#  model evaluation

In [ ]:
r2 = r2_score(y_test, y_pred)
r2                                    # r2 = 0.88622, means 82.662 %

In [ ]:
n = X_test.shape[0]
p = X_test.shape[1]

adjusted_r2 = 1 - ((1-r2) * (n-1)) / (n-p-1)
adjusted_r2                                  # adjusted r2 = 0.50463, means 50.463 %

In [ ]:
# First Model - r2 = 82.662 % ; adjusted_r2 = 50.463 %

- Second model

In [ ]:
X_final2 = X_encode[['enginesize', 'horsepower', 'carwidth', 'fuelsystem_2bbl', 'drivewheel_rwd', 'drivewheel_fwd', 'fuelsystem_mpfi', 
                     'cylindernumber_four', 'enginesize_category_medium']]      

In [ ]:
# data split 

X_train, X_test, y_train, y_test = train_test_split(X_final2, y, test_size=0.20, random_state = 42) 

In [ ]:
# training

model2 = LinearRegression()
model2.fit(X_train, y_train)

In [ ]:
# testing

y_pred = model2.predict(X_test)
y_pred

In [ ]:
# model evaluation

In [ ]:
r2 = r2_score(y_test, y_pred)
r2                             # r2 = 0.80665, means 80.665 %

In [ ]:
n = X_test.shape[0]
p = X_test.shape[1]

adjusted_r2 = 1 - ((1-r2) * (n-1)) / (n-p-1)
adjusted_r2                                      # adjusted_r2 = 0.75051, means 75.501 %

- Second Model - r2 = 80.665 % ; adjusted_r2 = 75.501 % (low bias, low variance)
- First Model - r2 = 82.662 % ; adjusted_r2 = 50.463 % (low bias, high variance - overfitting)
- so Second Model is better